# KernelGYM correctness-check audit with planted Triton mutants

> **UNTESTED.** This notebook has been written but never run. It needs a CUDA GPU (Kaggle T4 or P100),
> and the author's machine has none. Expect to fix small API mismatches on the first run.

What it does:

1. Clones [KernelGYM](https://github.com/hkust-nlp/KernelGYM) at a pinned commit and this repo
   ([triton_tileproof](https://github.com/rajul-kk/triton_tileproof)).
2. Wraps every reference kernel and every mutant from `triton_equiv.mutate` (43 mutants over 5 kernels) as a
   KernelGYM `ModelNew` module.
3. Runs each one through KernelGYM's `kernel_simple` correctness check
   (`kernelgym/toolkit/kernel_simple/toolkit.py`, `KernelSimpleToolkit.evaluate`). It calls the toolkit in-process
   with the `kernelbench` backend adapter, the same adapter that `kernelgym/workflow/kernel_simple.py` selects. The Redis/
   scheduler layer is bypassed.
4. Records which mutants the check accepts and joins that with the oracle labels in `results/labels.json`
   (`oob`, `inequivalent`, `numerical`, `benign`).

Three tolerance settings per case:
* `explicit_default`: `rtol=1e-4, atol=1e-5`, the values the toolkit falls back to.
* `kernelbench`: `rtol=atol=1e-2`.
* `unspecified`: no `rtol`/`atol` in the case. **Reading the code** (not run), `_normalize_case` stores
  `rtol=None`, so `case.get("rtol", 1e-4)` returns `None`, `torch.allclose` raises, and the comparison
  returns `False` for every kernel. The reference-kernel control below tests that reading.

In [ ]:
# Kaggle: Settings -> Accelerator -> GPU. Internet must be on for the clones.
import os, subprocess, sys
os.environ["TRITON_INTERPRET"] = "0"   # real GPU; triton_equiv.harness defaults to the interpreter otherwise
KGYM_COMMIT = "3a84417f8c0efaadb215ef638b37d12e71ed20f3"   # KernelGYM main as read on 2026-11-06
subprocess.run(["git", "clone", "-q", "https://github.com/hkust-nlp/KernelGYM.git", "/kaggle/working/KernelGYM"], check=False)
subprocess.run(["git", "-C", "/kaggle/working/KernelGYM", "checkout", "-q", KGYM_COMMIT], check=True)
subprocess.run(["git", "clone", "-q", "https://github.com/rajul-kk/triton_tileproof.git", "/kaggle/working/triton_tileproof"], check=False)
# KernelGYM's requirements.txt pulls a whole server stack; install only what the toolkit imports.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic", "pydantic-settings==2.10.1",
                "structlog", "z3-solver"], check=True)
sys.path[:0] = ["/kaggle/working/KernelGYM", "/kaggle/working/triton_tileproof/src"]

In [ ]:
import json, torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name(0))
from kernelgym.backend import get_backend
from kernelgym.toolkit.kernel_simple.toolkit import KernelSimpleToolkit
from triton_equiv.kernels import KERNELS
from triton_equiv.mutate import mutants
from triton_equiv import harness
labels = {l["id"]: l for l in json.load(open("/kaggle/working/triton_tileproof/results/labels.json"))["labels"]}
backend = get_backend("kernelbench")
toolkit = KernelSimpleToolkit()

## Wrap a kernel source as a KernelGYM `ModelNew`

One module per (kernel source, size). The size and launch grid are baked in. `forward` takes the input buffers
as flat tensors, allocates outputs with `torch.empty` (as generated kernels usually do) and returns them.

In [ ]:
DT = {"float16": "torch.float16", "float32": "torch.float32"}

def model_code(spec, source, size):
    L = spec.make_launch(**size)
    ins = [b for b in L.buffers if b not in L.outputs]
    outs = list(L.outputs)
    args = ", ".join(f"{k}={v}" for k, v in L.args.items())
    alloc = "\n".join(f"        {b} = torch.empty({L.buffers[b]}, dtype={DT[L.dtype(b)]}, device={ins[0]}.device)"
                       for b in outs)
    ret = outs[0] if len(outs) == 1 else "(" + ", ".join(outs) + ")"
    return f"""
import torch
import torch.nn as nn
import triton
import triton.language as tl
{source}

class ModelNew(nn.Module):
    def forward(self, {", ".join(ins)}):
{alloc}
        {spec.fn_name}[{tuple(L.grid)}]({args})
        return {ret}
"""

def cases_for(spec, size, policy, trials=3):
    L = spec.make_launch(**size)
    out = []
    for seed in range(trials):
        inputs = harness.make_inputs(L, seed)                       # torch.randn, rounded to buffer dtype
        ref = harness.torch_reference(spec.name, L, inputs)[L.outputs[0]]
        case = {"name": f"seed{seed}", "inputs": [inputs[b] for b in L.buffers if b not in L.outputs],
                "outputs": ref}
        if policy == "explicit_default":
            case.update(rtol=1e-4, atol=1e-5)
        elif policy == "kernelbench":
            case.update(rtol=1e-2, atol=1e-2)
        out.append(case)                                            # "unspecified": no rtol/atol keys
    return out

def evaluate(spec, source, size, policy, task_id):
    task = {"task_id": task_id, "kernel_code": model_code(spec, source, size), "backend": "triton",
            "entry_point": "ModelNew", "device": "cuda:0", "run_correctness": True, "run_performance": False,
            "cases": cases_for(spec, size, policy)}
    r = toolkit.evaluate(task, backend=backend)
    return {"correctness": r.get("correctness"), "compiled": r.get("compiled"),
            "error": r.get("error_message"), "failed": r.get("metadata", {}).get("correctness_failed_cases")}

## Controls: every reference kernel must pass

If a reference fails under `explicit_default` or `kernelbench`, the wrapper or the tolerance is wrong, so stop
and fix that first. Under `unspecified`, the reading above predicts `correctness == False` for every reference.

In [ ]:
POLICIES = ("explicit_default", "kernelbench", "unspecified")
controls = {}
for name, spec in KERNELS.items():
    for kind, sizes in spec.checker_sizes.items():
        for p in POLICIES:
            controls[(name, kind, p)] = evaluate(spec, spec.source, sizes[0], p, f"{name}:ref:{kind}:{p}")
            print(name, kind, p, controls[(name, kind, p)])

## Plant every mutant

In [ ]:
rows = []
for name, spec in KERNELS.items():
    for m in mutants(name, spec.source):
        lab = labels[m.id]
        fam = harness._family(lab)
        for kind, sizes in spec.checker_sizes.items():
            for p in ("explicit_default", "kernelbench"):
                r = evaluate(spec, m.source, sizes[0], p, f"{m.id}:{kind}:{p}")
                rows.append({"id": m.id, "oracle": fam, "size_kind": kind, "policy": p, **r})
                print(m.id, fam, kind, p, r["correctness"], (r["error"] or "")[:80])

## What slips through

A mutant "slips through" when KernelGYM reports `correctness == True` although the oracle label is `oob`,
`inequivalent` or `numerical:precision`. `numerical:reorder` and `benign` acceptances are expected.
On a GPU, out-of-bounds stores can corrupt other allocations or fault, which the CPU interpreter run never sees.
Compare with `results/checker_audit.json` (same mutants, CPU interpreter, our allclose harness).

In [ ]:
from collections import Counter
table = Counter((r["oracle"], r["size_kind"], r["policy"], r["correctness"]) for r in rows)
for k in sorted(table, key=str):
    print(k, table[k])
slipped = [r for r in rows if r["correctness"] is True
           and r["oracle"] in ("oob", "inequivalent", "numerical:precision")]
print(len(slipped), "accepted faulty mutants")
json.dump({"kernelgym_commit": KGYM_COMMIT, "gpu": torch.cuda.get_device_name(0),
           "torch": torch.__version__, "triton": triton.__version__,
           "controls": {"|".join(k): v for k, v in controls.items()}, "rows": rows},
          open("/kaggle/working/kernelgym_audit.json", "w"), indent=1, default=str)